# <font color=firebrick>Econometrics — M2 MBFA</font>
## Session 5 / 8 (CM) — Panel Data (Simplified)

**Prerequisites:** Sessions 1-4 — the OLS estimator, multivariate regression, dummy variables, and omitted variable bias.

**Format.** Theory first, kept deliberately light on new calculus: the main idea of this session reuses tools you already have (dummy variables, OLS) rather than introducing new ones.

**Today's objectives:**
- Understand what panel data is, and why it is the natural structure for financial data - many assets or portfolios, observed over many periods.
- Understand why simply pooling all observations into one OLS regression is usually biased, and why.
- Learn the **fixed-effects** fix, in two equivalent forms, and see it change a real, famous result.

Practice exercises for this session are provided in the companion TD notebook, applied to real trading data.

---


# 1. Why panel data?

The datasets we used so far were of a **cross-sectional** nature: individuals or firms were observed only once (`ceosal1`, `ceosal2`, `loanapp`...). Another very common structure in finance is that of **panel data**: several individuals (or assets, portfolios, firms) are observed on multiple occasions over time. For example, consider the daily values of trading portfolios, or the monthly returns of numerous shares over several years: an entity dimension ($i$) *and* a time dimension ($t$) simultaneously.

**What does the $t$ dimension actually buy us?** It is tempting to think of panel data as simply "more observations" - $N\times T$ rows instead of $N$. But repeating observations on the *same* individual is qualitatively different from adding *new*, different individuals. To see why, it helps to distinguish two sources of variation in any variable $x_{it}$:
- **Between-individual variation**: differences between trader $i$ and trader $j$ - e.g. one is generally more risk-seeking than the other. A single cross-section only ever sees this kind of variation.
- **Within-individual variation**: how a *given* trader $i$'s own behavior changes from one period to the next - e.g. that same trader taking on more risk this month than last. This kind of variation exists only because we observe the *same* individual repeatedly - it is genuinely new information that only panel data provides.

Why does this distinction matter so much? Anything that is *stable* about trader $i$ - skill, risk tolerance, general ability - shows up in the *between* variation, but never in the *within* variation, precisely because it does not change from one $t$ to the next for that individual. This is the seed of the whole session: if we could somehow look **only** at the within variation - each individual compared only to their own average - whatever is stable about them would disappear from the picture automatically, without our ever having to measure it. Making that idea precise is the mathematical thread running through Sections 2-3 below.

Concretely: if we only ever see one portfolio, we cannot tell whether it did well because of a skilled strategy (something *about that trader*, i.e. between-variation) or a good day for the market (something *about that day*, affecting everyone). With many traders observed on the same days, we can start to tell the two apart.


# 2. The panel data model

For individual $i$ ($i=1,\dots,N$) observed at time $t$ ($t=1,\dots,T$), we write:
$$
y_{it} = \beta_0+\beta_1x_{it}+a_i+\varepsilon_{it}.
$$
Everything here is familiar except one new term, $a_i$. It represents everything about individual $i$ that is fixed over time and affects $y_{it}$, but that we do not observe: innate trading skill, risk tolerance, ability, management quality - whatever it is, as long as it does not change (much) over the period we study. $a_i$ is called the **individual (fixed) effect**. The remaining error $\varepsilon_{it}$ is the usual idiosyncratic noise, specific to individual $i$ *and* time $t$.

Why single out $a_i$ instead of leaving it inside $\varepsilon_{it}$, as we always did before? Because panel data gives us a way to deal with it that cross-sectional data never could - as the next section shows, first by seeing what goes wrong if we don't.

##### Why pooling everything into one OLS is usually biased

The simplest thing to try is to ignore the panel structure entirely: stack every $(i,t)$ observation together and run one ordinary OLS regression of $y_{it}$ on $x_{it}$ - called **pooled OLS**. This silently treats $a_i$ as part of the error term:
$$
y_{it} = \beta_0+\beta_1x_{it}+\underbrace{(a_i+\varepsilon_{it})}_{\text{treated as "the error"}}.
$$
Recall the zero conditional mean assumption behind unbiasedness (Session 2): we need the *whole* error term to be uncorrelated with $x_{it}$. If $a_i$ - unobserved skill, say - is correlated with $x_{it}$ - say, a risk-taking choice more skilled traders happen to make more often - then this assumption fails, for exactly the reason Session 3 already worked out in detail: $a_i$ is an **omitted variable**, correlated with the regressor we kept. Pooled OLS is biased by precisely $\beta_a\delta$, where $\beta_a$ is the effect of $a_i$ on $y_{it}$ and $\delta$ is the correlation between $a_i$ and $x_{it}$ - the exact same formula as Section 0 of Session 3, with $a_i$ playing the role of the omitted variable.

The practical problem is that $a_i$ is, by construction, **unobserved** - we cannot just add it as a control the way we added `lsales` back in Session 3. We need another way to get rid of it. That is what fixed effects do.

##### A note on random effects

You may encounter another name in the panel data literature: the **random-effects (RE)** estimator. It uses the same model, $y_{it}=\beta_0+\beta_1x_{it}+a_i+\varepsilon_{it}$, but under a much stronger assumption: that $a_i$ is **uncorrelated** with $x_{it}$ - a "random", not systematic, source of individual variation. If that assumption holds, RE is more efficient than FE (it does not throw away the *between*-individual variation the way the within transformation does). But it is also a strong, often implausible assumption in observational data: if the reasons someone joins a union or takes more portfolio risk are related to their unobserved ability or risk tolerance - exactly the concern that motivated fixed effects in Section 3 - RE is biased for the same reason pooled OLS was. For this reason, **fixed effects is the safer default** whenever you are not confident $a_i$ is unrelated to your regressors, and is the estimator we focus on in this course.

---



# 3. Fixed effects: getting rid of $a_i$ without observing it

##### First way: a dummy for every individual

Section 3 of the Session 3 notebook already showed that including a full set of dummy variables absorbs a systematic differences across categories (there, sectors). Nothing stops us from doing the same thing here, with one dummy **per individual**:
$$
y_{it} = \beta_0+\beta_1x_{it}+\sum_{j=1}^{N} \gamma_j\,\mathbb{1}\{i=j\}+\varepsilon_{it}.
$$
This is called the **LSDV** estimator (Least Squares Dummy Variables). It works, and $\hat\beta_1$ is unbiased - $a_i$ has effectively been controlled for, one dummy at a time - but with hundreds or thousands of individuals, adding that many dummy columns is wasteful.

##### Second way: subtract the individual's own average

There is a shortcut that gives the *exact same* $\hat\beta_1$ without ever creating a single dummy. Average the model over time for each individual $i$ (using a bar to denote an individual-specific average over $t$):
$$
\bar y_i = \beta_0+\beta_1\bar x_i+a_i+\bar\varepsilon_i.
$$
Now subtract this from the original equation, observation by observation:
$$
y_{it}-\bar y_i = \beta_1(x_{it}-\bar x_i) + (\varepsilon_{it}-\bar\varepsilon_i).
$$
$\beta_0$ and $a_i$ - anything constant for individual $i$ - **cancel out exactly**, because we subtracted their own average. We never had to know $a_i$'s value to eliminate it. This is called the **within transformation** (we only use variation *within* each individual over time), and running OLS on these demeaned variables is the **fixed-effects (FE) estimator**. It is numerically identical to LSDV, and it is what statistical software actually computes, since it avoids creating hundreds of dummy columns.

##### Consequences for standard errors: a recap, and a subtlety

Once the data is demeaned, everything from Session 2 applies exactly as before: the *same* variance formula, $\widehat{\text{Var}}(\hat\beta)=\hat\sigma^2(\mathbf{X}^\top\mathbf{X})^{-1}$, the *same* t-test logic, with $\hat\sigma^2=\frac{1}{\text{df}}\sum_i\hat\varepsilon_i^2$ - the mechanics genuinely do not change. The one subtlety is the **degrees of freedom**. Recall from Session 2 that $\text{df}=N-p$: the number of observations minus the number of parameters actually estimated. In LSDV this is honest by construction - $p$ counts every dummy column, so $\text{df}=NT-N-k$ (with $k$ the number of regressors of interest). But if you run OLS *directly* on the demeaned variables, the regression only "sees" $k$ regressors - it has no way to know that $N$ individual means were silently subtracted beforehand, so it reports $\text{df}=NT-k$ instead: too generous by exactly $N$.

Does this matter in practice? Let's check, on the same `union` coefficient as in Section 4 below.


In [ ]:
# This cell reuses `wagepan_dm` and the fitted models from Section 4 below,
# so it only makes sense to run *after* that section - it is placed here for
# the discussion, but Python will execute the whole notebook top to bottom,
# so we recompute everything needed directly.
import numpy as np
import statsmodels.formula.api as smf
from wooldridge import load_data

wagepan_check = load_data.data('wagepan')
demean_cols = ['lwage', 'married', 'union', 'exper', 'expersq']
means_check = wagepan_check.groupby('nr')[demean_cols].transform('mean')
for c in demean_cols:
    wagepan_check[c + '_dm'] = wagepan_check[c] - means_check[c]

fe_naive = smf.ols('lwage_dm ~ married_dm + union_dm + exper_dm + expersq_dm - 1', data=wagepan_check).fit()
fe_lsdv_check = smf.ols('lwage ~ married + union + exper + expersq + C(nr)', data=wagepan_check).fit()

N = wagepan_check['nr'].nunique()
NT = len(wagepan_check)
k = 4  # married, union, exper, expersq

print("Naive within  : df_resid =", int(fe_naive.df_resid), " SE(union) =", fe_naive.bse['union_dm'])
print("Correct (LSDV): df_resid =", int(fe_lsdv_check.df_resid), " SE(union) =", fe_lsdv_check.bse['union'])

# Manual correction: rescale the naive SE by sqrt(naive_df / correct_df)
correct_df = NT - N - k
corrected_se = fe_naive.bse['union_dm'] * np.sqrt(fe_naive.df_resid / correct_df)
print("Naive SE, manually corrected:", corrected_se)


The naive standard error understates the correct one by about $7\%$ here (point estimates are identical either way - only the standard error, and hence the t-statistic and p-value, are affected). In this example the conclusion (union membership matters, statistically) does not change - the impact really is minor - but it is not *zero*, and in a borderline case it could flip a conclusion. This is why dedicated panel-data software applies this correction automatically, and why LSDV, despite being computationally wasteful, remains a useful sanity check: it gets the degrees of freedom right by construction, with nothing extra to remember.

With the machinery now behaving exactly as expected, we can already anticipate where this is going: comparing many portfolios' risk-adjusted performance over many days, controlling simultaneously for *who* is trading (individual fixed effects) and *which day* it is (time fixed effects, next) - exactly the two-way fixed-effects setting of the TD.

##### Beyond one dimension: two-way fixed effects

Everything above removes $a_i$, individual heterogeneity. In financial applications there is often a *second* source of unobserved heterogeneity to worry about: shocks common to a given **period** - a broad market rally or crash affecting every asset or portfolio on the same day, regardless of who holds it. Left uncontrolled, a good market day can look like individual skill.

The fix is the same trick, applied a second time: add a full set of **time dummies** (one per period) alongside the individual fixed effects - or equivalently, demean by *both* the individual average and the period average (with a small correction for the overall average, to avoid removing the variation twice). This is called a **two-way fixed-effects** model:
$$
y_{it} = \beta_0+\beta_1x_{it}+a_i+d_t+\varepsilon_{it}.
$$
In `wagepan`, the year dummies `d81`-`d87` are already provided for exactly this purpose - adding them to the fixed-effects regression above would additionally strip out any wage growth common to *all* workers in a given year (inflation, aggregate labor-market conditions), isolating even more precisely what is specific to each worker. We leave this as an exercise in the TD, applied to real trading data, where the two dimensions are natural: **who** is trading, and **which day** it is.

---



# 4. Application: does union membership raise wages?

We use `wagepan` (`wooldridge`) - a classic panel: $545$ workers, each observed every year from $1980$ to $1987$ ($N=545$, $T=8$, $4\,360$ observations in total). The question: controlling for unobserved, time-invariant worker characteristics (ability, motivation...), does being in a union raise wages?


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from wooldridge import load_data

wagepan = load_data.data('wagepan')
print("Individuals:", wagepan['nr'].nunique(), " | Years:", sorted(wagepan['year'].unique()))
wagepan[['nr', 'year', 'lwage', 'married', 'union', 'exper']].head(8)


## Pooled OLS first (ignoring the panel structure)


In [ ]:
pooled = smf.ols('lwage ~ married + union + exper + expersq', data=wagepan).fit()
print(pooled.summary())


**Task.** Read off the coefficients on `married` and `union`. Recall `lwage` is a log, so these are (approximately) percentage wage premia. Do these numbers strike you as economically large?

## Now the within transformation, by hand

**Task.** Complete the code below: for each variable, subtract its individual-specific mean (`groupby('nr').transform('mean')`), then run OLS on the demeaned variables, **without an intercept** (the intercept is a constant, so it is demeaned away to exactly zero - `- 1` in the formula removes it).


In [ ]:
demean_cols = ['lwage', 'married', 'union', 'exper', 'expersq']

# TODO: for each column, subtract its individual-specific (nr) mean, storing the result in f"{col}_dm"
wagepan_dm = wagepan.copy()
individual_means = wagepan.groupby('nr')[demean_cols].transform('mean')
for c in demean_cols:
    wagepan_dm[c + '_dm'] = ...

fe_manual = smf.ols('lwage_dm ~ married_dm + union_dm + exper_dm + expersq_dm - 1', data=wagepan_dm).fit()
print(fe_manual.params[['married_dm', 'union_dm']])
assert np.isclose(fe_manual.params['union_dm'], 0.0821, atol=0.001), "Doesn't match the expected FE union coefficient - check your demeaning."
print("Self-check passed!")


**Task.** Compare the fixed-effects coefficients on `married` and `union` to the pooled OLS ones. Both should have shrunk considerably. What does this tell you about workers who marry or join a union - are they a *random* subset of the workforce, or systematically different in some unobserved way (e.g. more stable jobs, more experience-per-year) that pooled OLS was wrongly crediting to marriage/union status itself?

## Checking the other route: LSDV

**Task.** Confirm that adding one dummy per worker (`C(nr)`) gives the *exact same* coefficients on `married` and `union` as the within transformation above - the two routes described in Section 4 are truly equivalent, not just similar. (This regression has hundreds of dummy coefficients - we only print the two we care about.)


In [ ]:
fe_lsdv = smf.ols('lwage ~ married + union + exper + expersq + C(nr)', data=wagepan).fit()
print(fe_lsdv.params[['married', 'union']])


##### A practical reality check: building your own panel data

Everything above assumed a clean, ready-made panel - `wagepan` is a carefully curated academic dataset, the product of a great deal of unseen data work. Building your **own** panel data - for a course project, say - is considerably more ambitious than the theory alone suggests. The within transformation implicitly assumes that $x_{it}$ *means the same thing* for every $i$ and every $t$: the same variable, measured the same way, at comparable moments in time. In practice this requires **standardizing** data collection from the very start (deciding what to record, and how, before you start recording it), and an ongoing **cleaning and updating** pipeline - handling missing observations (an individual absent on some dates), definitions or formats that drift over time, and keeping the data *actualisées* (kept up to date), rather than a one-off snapshot. Skipping this groundwork does not make the panel-data machinery wrong - but it can silently make $x_{it}$ not really comparable across $i$ or $t$, which quietly breaks the very idea the within transformation relies on.

---


## Summary

| Step | Question | Tool |
|---|---|---|
| Section 1 | What does the $t$ dimension add? | Between- vs within-individual variation; only repeated observations reveal the latter |
| Section 2 | What is $a_i$, and why is pooled OLS biased? | Unobserved, time-invariant heterogeneity; acts as an omitted variable (Session 3's result) |
| Section 3 | How do we remove $a_i$, and what changes in the calculations? | Dummy per individual (LSDV) or the within transformation - identical coefficients, but the degrees of freedom must be corrected by hand outside LSDV |
| Section 4 | Does it matter in practice? | Yes: the union/marriage wage premium shrinks by more than half once fixed effects are added |

Practice exercises applying all of this to real trading data are provided in the companion TD notebook.
